In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("blood_donation.csv")

# View basic info
print(df.head())
print("\n")
print(df.info())
print("\n")
print(df.describe(include='all'))

    Donor_ID       Full_Name  Gender  Age Blood_Group  Contact_Number  \
0  DNR000001  Sangeeta Menon  Female   38          O+      1819600042   
1  DNR000002      Meena Iyer  Female   49          B+       265423420   
2  DNR000003      Priya Nair  Female   29          B+      1849593012   
3  DNR000004    Vijay Kapoor    Male   29          O+      3419283185   
4  DNR000005      Rahul Iyer    Male   27          A+      6413953676   

                          Email                City           State Country  \
0  sangeeta.menon8280@gmail.com             Kolkata     West Bengal   India   
1      meena.iyer6225@gmail.com              Jaipur       Rajasthan   India   
2      priya.nair4742@gmail.com             Gurgaon         Haryana   India   
3    vijay.kapoor4423@gmail.com  Thiruvananthapuram          Kerala   India   
4      rahul.iyer2341@gmail.com              Bhopal  Madhya Pradesh   India   

  Last_Donation_Date  Total_Donations Eligible_for_Donation Medical_Condition  \
0    

In [2]:
import pandas as pd
import numpy as np

# Reload dataset fresh
df = pd.read_csv("blood_donation.csv")

# 1️⃣ Replace "Never" with NaN
df["Last_Donation_Date"] = df["Last_Donation_Date"].replace("Never", np.nan)

# 2️⃣ Convert date columns to datetime
df["Last_Donation_Date"] = pd.to_datetime(df["Last_Donation_Date"], dayfirst=True, errors='coerce')
df["Registration_Date"] = pd.to_datetime(df["Registration_Date"], dayfirst=True, errors='coerce')

# 3️⃣ Check results
print(df[["Last_Donation_Date", "Registration_Date"]].head())
print(df[["Last_Donation_Date", "Registration_Date"]].isnull().sum())

  Last_Donation_Date Registration_Date
0         2025-10-07        2021-07-02
1         2020-11-08        2023-03-03
2         2025-04-12        2015-10-15
3         2025-02-21        2022-05-09
4         2024-04-18        2022-07-13
Last_Donation_Date    576
Registration_Date       0
dtype: int64


In [3]:
# Get today's date
today = pd.Timestamp.today()

# 1 Create days_since_last_donation
df["days_since_last_donation"] = (today - df["Last_Donation_Date"]).dt.days

# 2️ For donors who never donated, replace NaN with days since registration
df["days_since_last_donation"] = df["days_since_last_donation"].fillna(
    (today - df["Registration_Date"]).dt.days
)

# 3️ Check results
print(df[["Last_Donation_Date", "days_since_last_donation"]].head(10))
print("\nAny remaining nulls?")
print(df["days_since_last_donation"].isnull().sum())

  Last_Donation_Date  days_since_last_donation
0         2025-10-07                     147.0
1         2020-11-08                    1941.0
2         2025-04-12                     325.0
3         2025-02-21                     375.0
4         2024-04-18                     684.0
5         2023-12-07                     817.0
6         2024-08-18                     562.0
7         2025-06-28                     248.0
8         2025-08-13                     202.0
9         2023-11-16                     838.0

Any remaining nulls?
0


In [6]:
# Create required donation interval based on gender
df["required_interval"] = df["Gender"].apply(
    lambda x: 90 if x.lower() == "male" else 120
)

# Preview
print(df[["Gender", "required_interval"]].head())

   Gender  required_interval
0  Female                120
1  Female                120
2  Female                120
3    Male                 90
4    Male                 90


In [7]:
# Calculate days until eligible
df["days_until_eligible"] = df["required_interval"] - df["days_since_last_donation"]

# If already eligible, set to 0
df["days_until_eligible"] = df["days_until_eligible"].apply(lambda x: 0 if x < 0 else x)

print(df[["days_since_last_donation", "required_interval", "days_until_eligible"]].head())

   days_since_last_donation  required_interval  days_until_eligible
0                     147.0                120                  0.0
1                    1941.0                120                  0.0
2                     325.0                120                  0.0
3                     375.0                 90                  0.0
4                     684.0                 90                  0.0


Compare Rule Eligibility vs Actual Label

In [8]:
# Create rule-based eligibility
df["eligible_by_interval_rule"] = df["days_since_last_donation"] >= df["required_interval"]

# Convert original label to boolean
df["Eligible_for_Donation_binary"] = df["Eligible_for_Donation"].map({"Yes": 1, "No": 0})

# Compare mismatch
mismatch = df[df["eligible_by_interval_rule"] != df["Eligible_for_Donation_binary"]]

print("Total mismatches:", len(mismatch))

Total mismatches: 3586


Prepare Medical & Clinical Features Properly

In [9]:
# 1️ Fill missing medical condition
df["Medical_Condition"] = df["Medical_Condition"].fillna("None")

# 2️ Create medical condition flag
df["has_medical_condition"] = df["Medical_Condition"].apply(lambda x: 0 if x == "None" else 1)

# 3️ Create hemoglobin risk flag
df["low_hemoglobin"] = df["Hemoglobin_g_dL"].apply(lambda x: 1 if x < 12.5 else 0)

# 4️ Create low weight flag
df["low_weight"] = df["Weight_kg"].apply(lambda x: 1 if x < 50 else 0)

# Check distribution
print(df[["has_medical_condition", "low_hemoglobin", "low_weight"]].head())

   has_medical_condition  low_hemoglobin  low_weight
0                      0               0           0
1                      1               0           0
2                      1               0           0
3                      0               0           0
4                      0               0           0


Analyze What Drives Ineligibility

In [10]:
# Compare averages between eligible and not eligible
df.groupby("Eligible_for_Donation_binary")[
    ["Hemoglobin_g_dL", "Weight_kg", "days_since_last_donation", "Age"]
].mean()

,Hemoglobin_g_dL,Weight_kg,days_since_last_donation,Age
Eligible_for_Donation_binary,,,,
0,13.094029,66.66197,699.828683,33.883092
1,14.097693,68.77168,706.513872,33.906016


Discover the Actual Medical Rules

In [11]:
# Check minimum values among eligible donors
print("Minimum hemoglobin among eligible:")
print(df[df["Eligible_for_Donation_binary"] == 1]["Hemoglobin_g_dL"].min())

print("Minimum weight among eligible:")
print(df[df["Eligible_for_Donation_binary"] == 1]["Weight_kg"].min())

# Check maximum values among ineligible donors
print("Maximum hemoglobin among ineligible:")
print(df[df["Eligible_for_Donation_binary"] == 0]["Hemoglobin_g_dL"].max())

print("Maximum weight among ineligible:")
print(df[df["Eligible_for_Donation_binary"] == 0]["Weight_kg"].max())

Minimum hemoglobin among eligible:
12.5
Minimum weight among eligible:
50.0
Maximum hemoglobin among ineligible:
17.6
Maximum weight among ineligible:
102.3


Recreate the Full Rule Manually

In [12]:
df["reconstructed_rule"] = (
    (df["Hemoglobin_g_dL"] >= 12.5) &
    (df["Weight_kg"] >= 50) &
    (df["has_medical_condition"] == 0) &
    (df["days_since_last_donation"] >= df["required_interval"])
).astype(int)

# Compare with actual label
mismatch_full_rule = df[df["reconstructed_rule"] != df["Eligible_for_Donation_binary"]]

print("Total mismatches after full rule reconstruction:", len(mismatch_full_rule))

Total mismatches after full rule reconstruction: 34


Select Features

In [13]:
# Define feature columns
features = [
    "Hemoglobin_g_dL",
    "Weight_kg",
    "Age",
    "days_since_last_donation",
    "has_medical_condition",
    "low_hemoglobin",
    "low_weight"
]

# If Gender exists, include it
if "Gender" in df.columns:
    features.append("Gender")

X = df[features]
y = df["Eligible_for_Donation_binary"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (10000, 8)
Target shape: (10000,)


Encode Gender

In [14]:
X = pd.get_dummies(X, drop_first=True)

Final Clean Modeling Dataset Creation

In [15]:
# ==============================
# STEP 10: FINAL MODEL DATASET
# ==============================

import pandas as pd

# ------------------------------
# 1. Define Final Modeling Features
# ------------------------------

final_features = [
    "Hemoglobin_g_dL",
    "Weight_kg",
    "Age",
    "days_since_last_donation",
    "has_medical_condition",
    "low_hemoglobin",
    "low_weight"
]

# Include Gender if it exists
if "Gender" in df.columns:
    final_features.append("Gender")

# ------------------------------
# 2. Create Clean Modeling Dataset
# ------------------------------

df_model = df[final_features + ["Eligible_for_Donation_binary"]].copy()

print("Final dataset shape:", df_model.shape)
print("\nPreview of cleaned dataset:")
print(df_model.head())

# ------------------------------
# 3. Encode Categorical Variables
# ------------------------------

if "Gender" in df_model.columns:
    df_model = pd.get_dummies(df_model, columns=["Gender"], drop_first=True)
    print("\nGender encoded using one-hot encoding.")

# ------------------------------
# 4. Verify No Data Leakage Columns
# ------------------------------

interval_cols = [col for col in df_model.columns if "interval" in col.lower()]
rule_cols = [col for col in df_model.columns if "rule" in col.lower()]

print("\nInterval-related columns found:", interval_cols)
print("Rule-related columns found:", rule_cols)

# ------------------------------
# 5. Save Clean Dataset
# ------------------------------

df_model.to_csv("cleaned_blood_donor_model_dataset.csv", index=False)

print("\nCleaned dataset saved as 'cleaned_blood_donor_model_dataset.csv'")

Final dataset shape: (10000, 9)

Preview of cleaned dataset:
   Hemoglobin_g_dL  Weight_kg  Age  days_since_last_donation  \
0             13.6       66.6   38                     147.0   
1             14.0       70.8   49                    1941.0   
2             12.5       73.4   29                     325.0   
3             14.8       57.9   29                     375.0   
4             17.1       74.0   27                     684.0   

   has_medical_condition  low_hemoglobin  low_weight  Gender  \
0                      0               0           0  Female   
1                      1               0           0  Female   
2                      1               0           0  Female   
3                      0               0           0    Male   
4                      0               0           0    Male   

   Eligible_for_Donation_binary  
0                             1  
1                             0  
2                             0  
3                             1  

Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))